# 🎙️ DDAA Pipeline (Kaggle Edition)

**Setup Instructions:**
1.  **Settings (Right Sidebar):**
    *   Internet: **ON** 🌍
    *   Accelerator: **GPU P100** 🚀
    *   Persistence: **Files only** (Optional)

**Workflow:**
1.  Clone Repository
2.  Install Dependencies
3.  Download Data (Direct from Mozilla)
4.  Run Pipeline
5.  **Save Output:** Important! You must download the output zip at the end.

## 1️⃣ Setup Repository

In [ ]:
import os
from getpass import getpass

# --- CONFIG ---
USERNAME = "charlesgrube-jpg"
REPO_NAME = "Data-Management-DDAA-KAN"
BRANCH = "feature/tts-vc-extraction"
# --------------

print("Enter GitHub Token (repo scope):")
token = getpass()
repo_url = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

# Clone into /kaggle/working (writable)
%cd /kaggle/working
if not os.path.exists(REPO_NAME):
    !git clone {repo_url} {REPO_NAME}
    %cd {REPO_NAME}
    !git checkout {BRANCH}
    print("✅ Repo Cloned!")
else:
    %cd {REPO_NAME}
    print("✅ Repo ready.")

## 2️⃣ Install Dependencies

In [ ]:
!apt-get update && apt-get install -y ffmpeg espeak-ng

# Install core libs
!pip install --no-deps git+https://github.com/facebookresearch/fairseq.git
!pip install --no-deps git+https://github.com/MisileLab/rvc-python-butter.git
!pip install -r requirements_colab.txt

## 3️⃣ Data & Model Setup
**Paste your Mozilla Signed URL in Option A.**

In [ ]:
# OPTION A: DIRECT DOWNLOAD
MOZILLA_URL = ""  # <-- Paste link here

# ------------------------------------------
TARGET_DIR = "/kaggle/working/Data-Management-DDAA-KAN/mozilla_cv_data"
import os
import shutil

def setup_data():
    if MOZILLA_URL.startswith("http"):
        print("☁️ Downloading Data...")
        os.makedirs(TARGET_DIR, exist_ok=True)
        !wget -O cv_corpus.tar.gz "{MOZILLA_URL}"
        print("📦 Extracting...")
        !tar -xzf cv_corpus.tar.gz -C {TARGET_DIR} --strip-components=1
        print("✅ Extraction Complete!")
    else:
        print("⚠️ No URL provided! If you uploaded a Dataset instead, adjust paths.")
    
    print("\n📥 Downloading RVC Models...")
    !python utilities/download_rvc_models.py

setup_data()

## 4️⃣ Run Pipeline

In [ ]:
import yaml

OUTPUT_PATH = "/kaggle/working/pipeline_output"

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

config['output']['base_dir'] = OUTPUT_PATH
config['source']['data_path'] = "mozilla_cv_data"
config['synthesis']['pick_strategy'] = "both"
config['synthesis']['vc_models'] = ["DuaLipa", "TaylorSwift", "EdSheeran", "KanyeWest"]
config['synthesis']['vc_device'] = "cuda:0"
config['codec_compression']['enabled'] = True

with open("config.yaml", "w") as f:
    yaml.dump(config, f)

!python run_pipeline.py

## 5️⃣ Zip Output for Download
Since Kaggle output is temporary, verify this runs and then look for `output_dataset.zip` in the **Output** tab.

In [ ]:
!zip -r output_dataset.zip {OUTPUT_PATH}